# Validasi Prediksi Harga Rumah Depok

Notebook bersih untuk mereproduksi evaluasi dalam `train_model.py`. Dataset berada di `dataset/Cleaned_Final_Rumah.csv`.

**Batas studi:** harga 100 juta–10 miliar; luas 20–1.500 m²; kamar tidur/mandi 1–15; garasi 0–10. Ini bukan evaluasi untuk seluruh pasar.

Sumber serta perizinan dataset memerlukan verifikasi.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT/'dataset').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT.resolve()))
from train_model import FEATURES, TARGET, in_scope, preprocess
print('Root:',ROOT.resolve())

Root: /mnt/data/Depok_ML_Validated


## 1. Profil dan filter cakupan data

In [2]:
raw = pd.read_csv(ROOT/'dataset'/'Cleaned_Final_Rumah.csv')
print('Raw:',raw.shape, '| Missing:',int(raw.isna().sum().sum()),'| Full duplicates:',int(raw.duplicated().sum()))
print('Lokasi unik:',raw['Lokasi'].nunique())
df = raw.loc[in_scope(raw)].reset_index(drop=True)
print('Dalam cakupan:',df.shape,'| Dikeluarkan:',len(raw)-len(df))
df[FEATURES+[TARGET]].describe(include='all')

Raw: (24561, 7) | Missing: 0 | Full duplicates: 0
Lokasi unik: 37
Dalam cakupan: (23292, 7) | Dikeluarkan: 1269


,Kamar Tidur,Kamar Mandi,Garasi,Luas Tanah,Luas Bangunan,Lokasi,Harga
count,23292.000000,23292.000000,23292.00000,23292.000000,23292.000000,23292,2.329200e+04
unique,NaN,NaN,NaN,NaN,NaN,37,NaN
top,NaN,NaN,NaN,NaN,NaN,Cinere,NaN
freq,NaN,NaN,NaN,NaN,NaN,4867,NaN
mean,3.312811,2.469560,1.10729,168.368839,149.958827,NaN,2.008565e+09
std,1.350270,1.332902,0.98053,171.539940,130.008795,NaN,1.799118e+09
min,1.000000,1.000000,0.00000,20.000000,20.000000,NaN,1.000000e+08
25%,2.000000,2.000000,0.00000,72.000000,67.000000,NaN,8.190000e+08
50%,3.000000,2.000000,1.00000,104.000000,102.000000,NaN,1.350000e+09
75%,4.000000,3.000000,2.00000,189.000000,200.000000,NaN,2.512500e+09


## 2. Group holdout untuk memisahkan kombinasi fitur identik

In [3]:
groups = pd.util.hash_pandas_object(df[FEATURES],index=False)
tr,te = next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(df,groups=groups))
assert not set(groups.iloc[tr]) & set(groups.iloc[te])
X_train,y_train=df.loc[tr,FEATURES],df.loc[tr,TARGET]
X_test,y_test=df.loc[te,FEATURES],df.loc[te,TARGET]
print('Train:',len(tr),'Test:',len(te))

Train: 18670 Test: 4622


## 3. Preprocessing fit hanya pada data latih dan perbandingan model

In [4]:
models={
 'Linear Regression':Pipeline([('prep',preprocess()),('model',LinearRegression())]),
 'Random Forest':Pipeline([('prep',preprocess()),('model',RandomForestRegressor(n_estimators=150,max_depth=20,min_samples_leaf=2,random_state=42,n_jobs=-1))])
}
results=[]
for name,m in models.items():
    m.fit(X_train,y_train)
    y_pred=m.predict(X_test)
    results.append({'Model':name,'MAE':mean_absolute_error(y_test,y_pred),'RMSE':np.sqrt(mean_squared_error(y_test,y_pred)),'R2':r2_score(y_test,y_pred)})
pd.DataFrame(results)

,Model,MAE,RMSE,R2
0,Linear Regression,5.166015e+08,8.620262e+08,0.764685
1,Random Forest,4.068168e+08,7.121016e+08,0.839420


## 4. Error model Random Forest menurut harga aktual

In [5]:
rf=models['Random Forest']
preds=rf.predict(X_test)
errs=pd.DataFrame({'actual':y_test.to_numpy(),'pred':preds})
errs['band']=pd.cut(errs.actual,[0,1e9,3e9,np.inf],labels=['≤ 1 miliar','(1, 3] miliar','> 3 miliar'])
errs.groupby('band',observed=True).apply(lambda g: pd.Series({'n':len(g),'MAE':mean_absolute_error(g.actual,g.pred)}),include_groups=False)

,n,MAE
band,,
≤ 1 miliar,1745.0,1.847534e+08
"(1, 3] miliar",2003.0,3.675856e+08
> 3 miliar,874.0,9.400901e+08


## 5. Contoh prediksi (bukan estimasi resmi)

In [6]:
example=pd.DataFrame([{'Kamar Tidur':3,'Kamar Mandi':2,'Garasi':1,'Luas Tanah':120,'Luas Bangunan':100,'Lokasi':'Cinere'}],columns=FEATURES)
print('Estimasi Rp:',round(float(rf.predict(example)[0])))

Estimasi Rp: 1675497196


## 6. Artefak model

Gunakan `python train_model.py` untuk menyimpan model dan `metrics.json` secara konsisten. Notebook ini sengaja tidak menimpa artefak.